# FluxNet Claim 3 — Colab CUDA run

Minimal research notebook for **FluxNet: Learning Capacity-Constrained Local Transport Operators for Conservative and Bounded PDE Surrogates** (arXiv:2602.01941, OpenReview `1KRpajnd6u`).

Before running: choose **Runtime → Change runtime type → GPU**, and upload `fluxnet_colab_fno_cuda_bundle.tar` to `MyDrive/fluxnet-colab/`. The notebook trains the projected-FNO baseline from scratch on CUDA, checkpoints every epoch to Drive, proves resume once, completes 100 epochs, evaluates, and packages raw outputs. It contains no test suite.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch

assert torch.cuda.is_available(), "Select a Colab GPU runtime before continuing."
print({"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)})


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/fluxnet-colab")
BUNDLE_PATH = DRIVE_ROOT / "fluxnet_colab_fno_cuda_bundle.tar"
BUNDLE_SHA256 = "f717ebee4fa3ce5ce3783c5e6883f39a79199b6d60d0a3c9a5bf38960aa4e050"
WORK_ROOT = Path("/content/fluxnet-repro")
DATA_DIR = WORK_ROOT / "data/shallow_water_attempt2"
OUTPUT_DIR = DRIVE_ROOT / "outputs/shallow_water_attempt2_cuda"
OFFICIAL_COMMIT = "ec0cafe3bb48cb7f2497723c5e12c6ebc518442c"
MODEL_NAMES = ["FNO_SW_Proj_box_mass_pf"]

assert BUNDLE_PATH.is_file(), f"Upload the bundle first: {BUNDLE_PATH}"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"bundle": str(BUNDLE_PATH), "output": str(OUTPUT_DIR), "models": MODEL_NAMES})


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_bundle_sha = sha256_file(BUNDLE_PATH)
assert actual_bundle_sha == BUNDLE_SHA256, (actual_bundle_sha, BUNDLE_SHA256)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(BUNDLE_PATH, "r") as archive:
    root = WORK_ROOT.resolve()
    for member in archive.getmembers():
        target = (WORK_ROOT / member.name).resolve()
        assert target == root or root in target.parents, f"Unsafe archive path: {member.name}"
    archive.extractall(WORK_ROOT)

official = WORK_ROOT / "official"
if not (official / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/Lan-zs/FluxNet.git", str(official)], check=True)
subprocess.run(["git", "-C", str(official), "checkout", "--detach", OFFICIAL_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(official), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
status = subprocess.run(["git", "-C", str(official), "status", "--porcelain", "--untracked-files=all"], check=True, capture_output=True, text=True).stdout
assert head == OFFICIAL_COMMIT and not status, (head, status)
print({"bundle_sha256": actual_bundle_sha, "official_commit": head, "data_manifest": str(DATA_DIR / 'manifest.json')})


In [ ]:
WRAPPER = WORK_ROOT / "repro/src/shallow_water_attempt2.py"
ENV = os.environ.copy()
ENV.update({
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONPATH": str(WORK_ROOT / "repro"),
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
})

def research_command(stage: str, max_new_epochs=None):
    command = [
        sys.executable, str(WRAPPER),
        "--stage", stage,
        "--device", "cuda",
        "--epochs", "100",
        "--batch-size", "16",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--block-pid", "999999999",
    ]
    for model in MODEL_NAMES:
        command.extend(["--model", model])
    if max_new_epochs is not None:
        command.extend(["--max-new-epochs", str(max_new_epochs)])
    return command

def run_research(stage: str, max_new_epochs=None):
    command = research_command(stage, max_new_epochs)
    print(" ".join(command))
    subprocess.run(command, check=True, env=ENV)


## Resume gate 1
Run exactly one epoch and persist it to Drive. On a reused Drive folder this advances exactly one additional epoch.


In [ ]:
run_research("train", max_new_epochs=1)
gate_checkpoint = OUTPUT_DIR / "models" / MODEL_NAMES[0] / "latest_checkpoint.pt"
gate1 = torch.load(gate_checkpoint, map_location="cpu", weights_only=False)
GATE1_EPOCH = int(gate1["completed_epochs"])
print({"gate1_epoch": GATE1_EPOCH, "best_validation_loss": gate1["best_validation_loss"]})


## Resume gate 2
Reload the Drive checkpoint in a fresh process and advance exactly one epoch. This proves checkpoint recovery before the long run.


In [ ]:
run_research("train", max_new_epochs=1)
gate2 = torch.load(gate_checkpoint, map_location="cpu", weights_only=False)
GATE2_EPOCH = int(gate2["completed_epochs"])
assert GATE2_EPOCH == GATE1_EPOCH + 1, (GATE1_EPOCH, GATE2_EPOCH)
assert len(gate2["history"]) == GATE2_EPOCH
assert gate2.get("cuda_rng_state_all"), "CUDA RNG state was not checkpointed"
print({"resume_verified": True, "from_epoch": GATE1_EPOCH, "to_epoch": GATE2_EPOCH})


## Complete the fixed 100-epoch run
This cell resumes from the verified Drive checkpoint. If Colab disconnects, remount Drive, rerun setup cells, and rerun this cell; it resumes from the latest completed epoch.


In [ ]:
run_research("train")
final_checkpoint = torch.load(gate_checkpoint, map_location="cpu", weights_only=False)
assert int(final_checkpoint["completed_epochs"]) == 100
assert len(final_checkpoint["history"]) == 100
print({"training_complete": True, "best_validation_loss": final_checkpoint["best_validation_loss"]})


## Evaluate and package raw research outputs


In [ ]:
run_research("evaluate")
RESULT_ARCHIVE = DRIVE_ROOT / "fluxnet_fno_cuda_results.tar.gz"
with tarfile.open(RESULT_ARCHIVE, "w:gz") as archive:
    archive.add(OUTPUT_DIR, arcname="shallow_water_attempt2_cuda")
print({"results": str(RESULT_ARCHIVE), "sha256": sha256_file(RESULT_ARCHIVE), "bytes": RESULT_ARCHIVE.stat().st_size})
